# 🧠 FracCaps Lite — CPU-Friendly Edition
### Same architecture, tuned to run without melting your machine

---

**What changed vs the original:**

| Setting | Original | Lite | Why |
|---------|----------|------|-----|
| `FRACTAL_DEPTH` | 3 | **2** | Halves recursive conv passes per branch |
| `BASE_CHANNELS` | 32 | **16** | 4× fewer feature map params in encoder |
| `PRIMARY_CAPS_NUM` | 32 | **16** | Smaller routing weight matrix W |
| `CLASS_CAPS_DIM` | 16 | **8** | Lighter decoder + routing |
| `ROUTING_ITERATIONS` | 3 | **2** | One fewer routing pass per forward step |
| `BATCH_SIZE` | 64 | **32** | Less memory per step, smaller gradient tensors |
| `EPOCHS` | 50 | **20** | Reach ~80% of full-run learning in 40% of the time |
| `NUM_WORKERS` | 2 | **0** | No background CPU threads fighting for cores |
| `pin_memory` | True | **False** | Avoid pinned RAM overhead without GPU |
| Dataset | CIFAR-10 | **MNIST** | 1-channel 28×28 vs 3-channel 32×32, ~3× faster per batch |
| AMP | — | **✓ autocast** | Mixed precision halves memory & speeds up CUDA ops |
| CPU throttle | — | **torch.set_num_threads(2)** | Caps CPU thread usage |

---

## Step 1 — Setup & Imports

In [ ]:
!pip install torch torchvision matplotlib seaborn scikit-learn tqdm --quiet

In [ ]:
import os, math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True

# ── CPU throttle — keeps your machine cool ────────────────────────────────────
# Increase to 4 if you have cores to spare and temps are OK
torch.set_num_threads(2)
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()   # Mixed precision only helps on CUDA
print(f"Device : {DEVICE}")
print(f"AMP    : {'enabled' if USE_AMP else 'disabled (CPU mode)'}")
print(f"Threads: {torch.get_num_threads()}")

### 1.1 — Lite Hyperparameter Config

> **Quick switch:** change `DATASET = 'CIFAR10'` below if you want the full dataset once temps are under control.

In [ ]:
class Config:
    # ── Data ──────────────────────────────────────────────────────────────────
    DATASET        = 'MNIST'    # ← 'CIFAR10' once thermals are OK
    IMG_SIZE       = 28         # MNIST=28; set to 32 for CIFAR10
    IMG_CHANNELS   = 1          # MNIST=1;  set to 3  for CIFAR10
    NUM_CLASSES    = 10
    BATCH_SIZE     = 32         # ↓ from 64 — smaller gradient tensors per step
    VAL_SPLIT      = 0.1
    NUM_WORKERS    = 0          # ↓ from 2 — no background threads stealing CPU

    # ── Fractal encoder — the biggest heat source ──────────────────────────────
    FRACTAL_DEPTH  = 2          # ↓ from 3 — cuts recursive conv passes roughly in half
    BASE_CHANNELS  = 16         # ↓ from 32 — 4× fewer feature map values

    # ── Capsule architecture ──────────────────────────────────────────────────
    PRIMARY_CAPS_DIM   = 8      # unchanged — minimum meaningful capsule
    PRIMARY_CAPS_NUM   = 16     # ↓ from 32 — smaller routing weight matrix
    CLASS_CAPS_DIM     = 8      # ↓ from 16 — lighter decoder
    ROUTING_ITERATIONS = 2      # ↓ from 3 — one fewer expensive routing pass

    # ── Training ──────────────────────────────────────────────────────────────
    EPOCHS         = 20         # ↓ from 50 — reaches ~80% of learning value
    LR             = 1e-3
    WEIGHT_DECAY   = 1e-4
    LR_MIN         = 1e-5
    RECON_WEIGHT   = 0.0005

    # ── Margin loss ───────────────────────────────────────────────────────────
    M_PLUS         = 0.9
    M_MINUS        = 0.1
    LAMBDA_MARGIN  = 0.5

    # ── I/O ───────────────────────────────────────────────────────────────────
    SAVE_PATH      = 'fraccaps_lite_best.pth'
    DATA_ROOT      = './data'

cfg = Config()
print("Lite config loaded ✓")
print(f"Dataset: {cfg.DATASET} | Batch: {cfg.BATCH_SIZE} | "
      f"Depth: {cfg.FRACTAL_DEPTH} | Channels: {cfg.BASE_CHANNELS} | Epochs: {cfg.EPOCHS}")

---
## Step 2 — Dataset Loading

In [ ]:
def get_transforms(train=True):
    """MNIST-compatible transforms. Lighter augmentation than CIFAR."""
    if cfg.DATASET == 'MNIST':
        normalize = transforms.Normalize((0.1307,), (0.3081,))
        if train:
            return transforms.Compose([
                transforms.RandomAffine(degrees=10, translate=(0.1, 0.1)),
                transforms.ToTensor(),
                normalize,
            ])
        return transforms.Compose([transforms.ToTensor(), normalize])
    else:
        # CIFAR-10 path
        normalize = transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))
        if train:
            return transforms.Compose([
                transforms.RandomCrop(32, padding=4),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                normalize,
            ])
        return transforms.Compose([transforms.ToTensor(), normalize])


# ── Load dataset ──────────────────────────────────────────────────────────────
DS = datasets.MNIST if cfg.DATASET == 'MNIST' else datasets.CIFAR10

full_train = DS(cfg.DATA_ROOT, train=True,  download=True, transform=get_transforms(True))
test_set   = DS(cfg.DATA_ROOT, train=False, download=True, transform=get_transforms(False))

val_size   = int(len(full_train) * cfg.VAL_SPLIT)
train_size = len(full_train) - val_size
train_set, val_set = random_split(full_train, [train_size, val_size],
                                   generator=torch.Generator().manual_seed(SEED))

# pin_memory=False — avoids pinned RAM cost on CPU-only machines
train_loader = DataLoader(train_set, batch_size=cfg.BATCH_SIZE, shuffle=True,
                          num_workers=cfg.NUM_WORKERS, pin_memory=False)
val_loader   = DataLoader(val_set,   batch_size=cfg.BATCH_SIZE, shuffle=False,
                          num_workers=cfg.NUM_WORKERS, pin_memory=False)
test_loader  = DataLoader(test_set,  batch_size=cfg.BATCH_SIZE, shuffle=False,
                          num_workers=cfg.NUM_WORKERS, pin_memory=False)

CLASS_NAMES = full_train.classes if hasattr(full_train, 'classes') else [str(i) for i in range(10)]
print(f"Train: {train_size:,}  |  Val: {val_size:,}  |  Test: {len(test_set):,}")
print(f"Classes: {CLASS_NAMES}")

---
## Step 3 — Fractal Encoder (Lite)

Same two-path fractal structure as the original. Depth reduced to 2 and channels halved — this is the single biggest heat reduction.

In [ ]:
class FractalBlock(nn.Module):
    """
    Path 1: conv → BN → ReLU  (direct)
    Path 2: recursive(conv → BN → ReLU → recursive(...))  (fractal)
    Output: element-wise mean. Depth=1 → plain conv block.
    """
    def __init__(self, in_channels: int, out_channels: int,
                 depth: int = 2, drop_path_prob: float = 0.1):
        super().__init__()
        self.depth = depth
        self.drop_path_prob = drop_path_prob

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        self.recursive = (
            FractalBlock(in_channels, out_channels, depth - 1, drop_path_prob)
            if depth > 1 else None
        )

    def forward(self, x):
        direct = self.conv(x)
        if self.recursive is None:
            return direct
        if self.training and torch.rand(1).item() < self.drop_path_prob:
            return direct
        return (direct + self.recursive(x)) * 0.5


class FractalBranch(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, depth=2):
        super().__init__()
        self.entry = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        self.fractal = FractalBlock(out_channels, out_channels, depth=depth)

    def forward(self, x):
        return self.fractal(self.entry(x))


class FractalEncoder(nn.Module):
    """
    Three parallel branches: fine (stride 1), mid (stride 2), coarse (stride 4).
    Mid and coarse are upsampled back to fine resolution and concatenated.
    """
    def __init__(self, in_channels=1, base_channels=16, depth=2):
        super().__init__()
        C = base_channels
        self.fine   = FractalBranch(in_channels, C,     stride=1, depth=depth)
        self.mid    = FractalBranch(in_channels, C * 2, stride=2, depth=depth)
        self.coarse = FractalBranch(in_channels, C * 4, stride=4, depth=depth)

        self.proj_fine   = nn.Conv2d(C,     C, 1)
        self.proj_mid    = nn.Conv2d(C * 2, C, 1)
        self.proj_coarse = nn.Conv2d(C * 4, C, 1)
        self.out_channels = C * 3

    def forward(self, x):
        H, W = x.shape[2], x.shape[3]
        f = self.proj_fine(self.fine(x))
        m = F.interpolate(self.proj_mid(self.mid(x)),    size=(H,W), mode='bilinear', align_corners=False)
        c = F.interpolate(self.proj_coarse(self.coarse(x)), size=(H,W), mode='bilinear', align_corners=False)
        return torch.cat([f, m, c], dim=1)


# ── Shape test ────────────────────────────────────────────────────────────────
enc = FractalEncoder(in_channels=cfg.IMG_CHANNELS,
                     base_channels=cfg.BASE_CHANNELS,
                     depth=cfg.FRACTAL_DEPTH)
dummy = torch.zeros(2, cfg.IMG_CHANNELS, cfg.IMG_SIZE, cfg.IMG_SIZE)
print(f"FractalEncoder output : {enc(dummy).shape}")
print(f"Encoder params        : {sum(p.numel() for p in enc.parameters()):,}")

---
## Step 4 — Primary Capsule Layer

In [ ]:
def squash(x, dim=-1):
    norm_sq = (x ** 2).sum(dim=dim, keepdim=True)
    norm    = norm_sq.sqrt()
    return (norm_sq / (1.0 + norm_sq)) * x / (norm + 1e-8)


class PrimaryCapsLayer(nn.Module):
    def __init__(self, in_channels, num_capsules=16, caps_dim=8,
                 kernel_size=3, stride=2):
        super().__init__()
        self.num_capsules = num_capsules
        self.caps_dim     = caps_dim
        self.conv = nn.Conv2d(in_channels, num_capsules * caps_dim,
                              kernel_size, stride=stride, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(num_capsules * caps_dim)

    def forward(self, x):
        out = self.bn(self.conv(x))
        B, _, H, W = out.shape
        out = out.view(B, self.num_capsules, self.caps_dim, H, W)
        out = out.permute(0,1,3,4,2).contiguous().view(B, -1, self.caps_dim)
        return squash(out)


prim_in = cfg.BASE_CHANNELS * 3
pcl = PrimaryCapsLayer(prim_in, cfg.PRIMARY_CAPS_NUM, cfg.PRIMARY_CAPS_DIM)
print(f"PrimaryCaps output: {pcl(torch.zeros(2, prim_in, cfg.IMG_SIZE, cfg.IMG_SIZE)).shape}")

---
## Step 5 — Dynamic Routing (2 iterations)

In [ ]:
class DynamicRouting(nn.Module):
    def __init__(self, num_in_caps, num_out_caps, in_dim, out_dim, routing_iters=2):
        super().__init__()
        self.num_in_caps  = num_in_caps
        self.num_out_caps = num_out_caps
        self.routing_iters = routing_iters
        self.W = nn.Parameter(
            torch.randn(1, num_in_caps, num_out_caps, out_dim, in_dim) * 0.1
        )

    def forward(self, x):
        B = x.size(0)
        x_exp = x.unsqueeze(2).unsqueeze(4).expand(B, self.num_in_caps,
                                                    self.num_out_caps, x.size(-1), 1)
        W_exp = self.W.expand(B, -1, -1, -1, -1)
        u_hat = torch.matmul(W_exp, x_exp).squeeze(-1)
        u_hat_detached = u_hat.detach()

        b = torch.zeros(B, self.num_in_caps, self.num_out_caps, 1, device=x.device)
        for i in range(self.routing_iters):
            c     = F.softmax(b, dim=2)
            votes = u_hat_detached if i < self.routing_iters - 1 else u_hat
            s     = (c * votes).sum(dim=1, keepdim=True)
            v     = squash(s, dim=-1)
            if i < self.routing_iters - 1:
                b = b + (u_hat_detached * v).sum(dim=-1, keepdim=True)

        return v.squeeze(1)


print("DynamicRouting defined ✓")

---
## Step 6 — Class Capsule Layer

In [ ]:
class ClassCapsuleLayer(nn.Module):
    def __init__(self, num_primary, num_classes, in_dim, out_dim, routing_iters=2):
        super().__init__()
        self.routing = DynamicRouting(num_primary, num_classes, in_dim, out_dim, routing_iters)

    def forward(self, x):
        caps    = self.routing(x)
        lengths = caps.norm(dim=-1)
        return caps, lengths


print("ClassCapsuleLayer defined ✓")

---
## Step 7 — Decoder

In [ ]:
class CapsDecoder(nn.Module):
    def __init__(self, caps_dim, num_classes, img_channels, img_size):
        super().__init__()
        self.caps_dim    = caps_dim
        self.num_classes = num_classes
        out_pixels       = img_channels * img_size * img_size

        # Slimmer decoder: 256 → 512 (was 512 → 1024)
        self.net = nn.Sequential(
            nn.Linear(caps_dim * num_classes, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, out_pixels),
            nn.Sigmoid(),
        )
        self.img_channels = img_channels
        self.img_size     = img_size

    def forward(self, caps, labels=None):
        B = caps.size(0)
        if labels is None:
            labels = caps.norm(dim=-1).argmax(dim=-1)
        mask   = F.one_hot(labels, self.num_classes).float().unsqueeze(-1)
        masked = (caps * mask).view(B, -1)
        recon  = self.net(masked)
        return recon.view(B, self.img_channels, self.img_size, self.img_size)


print("CapsDecoder defined ✓")

---
## Step 8 — Full FracCaps Lite Model

In [ ]:
class FracCaps(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.encoder = FractalEncoder(
            in_channels   = cfg.IMG_CHANNELS,
            base_channels = cfg.BASE_CHANNELS,
            depth         = cfg.FRACTAL_DEPTH,
        )
        enc_out_ch = cfg.BASE_CHANNELS * 3

        self.primary_caps = PrimaryCapsLayer(
            in_channels  = enc_out_ch,
            num_capsules = cfg.PRIMARY_CAPS_NUM,
            caps_dim     = cfg.PRIMARY_CAPS_DIM,
        )

        with torch.no_grad():
            dummy    = torch.zeros(1, enc_out_ch, cfg.IMG_SIZE, cfg.IMG_SIZE)
            n_primary = self.primary_caps(dummy).shape[1]
        self.n_primary = n_primary

        self.class_caps = ClassCapsuleLayer(
            num_primary   = n_primary,
            num_classes   = cfg.NUM_CLASSES,
            in_dim        = cfg.PRIMARY_CAPS_DIM,
            out_dim       = cfg.CLASS_CAPS_DIM,
            routing_iters = cfg.ROUTING_ITERATIONS,
        )

        self.decoder = CapsDecoder(
            caps_dim     = cfg.CLASS_CAPS_DIM,
            num_classes  = cfg.NUM_CLASSES,
            img_channels = cfg.IMG_CHANNELS,
            img_size     = cfg.IMG_SIZE,
        )

    def forward(self, x, labels=None):
        features        = self.encoder(x)
        prim_caps       = self.primary_caps(features)
        caps, lengths   = self.class_caps(prim_caps)
        recon           = self.decoder(caps, labels)
        return lengths, caps, recon


model = FracCaps(cfg).to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total:,}")
print(f"Trainable parameters : {trainable:,}")
print(f"Primary capsules     : {model.n_primary}")

# Forward pass smoke test
with torch.no_grad():
    x_t = torch.zeros(4, cfg.IMG_CHANNELS, cfg.IMG_SIZE, cfg.IMG_SIZE).to(DEVICE)
    l_t = torch.zeros(4, dtype=torch.long).to(DEVICE)
    lengths_t, caps_t, recon_t = model(x_t, l_t)
print(f"Forward pass OK — lengths:{lengths_t.shape} caps:{caps_t.shape} recon:{recon_t.shape}")

---
## Step 9 — Loss Functions

In [ ]:
class MarginLoss(nn.Module):
    def __init__(self, m_plus=0.9, m_minus=0.1, lam=0.5):
        super().__init__()
        self.m_plus = m_plus; self.m_minus = m_minus; self.lam = lam

    def forward(self, lengths, labels):
        T        = F.one_hot(labels, lengths.size(1)).float()
        loss_pos = T       * F.relu(self.m_plus  - lengths) ** 2
        loss_neg = (1 - T) * F.relu(lengths - self.m_minus) ** 2 * self.lam
        return (loss_pos + loss_neg).sum(dim=1).mean()


class FracCapsLoss(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.margin_loss = MarginLoss(cfg.M_PLUS, cfg.M_MINUS, cfg.LAMBDA_MARGIN)
        self.recon_w     = cfg.RECON_WEIGHT

    def forward(self, lengths, recon, labels, images):
        ml = self.margin_loss(lengths, labels)
        rl = F.mse_loss(recon, images)
        return ml + self.recon_w * rl, ml.item(), rl.item()


criterion = FracCapsLoss(cfg)
print("Loss functions defined ✓")

---
## Step 10 — Training Loop

Key additions vs original:
- **`torch.cuda.amp.autocast`** — mixed precision on CUDA (no-op on CPU)
- **`GradScaler`** — matches autocast for stable FP16 gradients
- **`torch.no_grad()` on val** — already present, confirms no extra graph overhead

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS, eta_min=cfg.LR_MIN)
scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP)   # no-op when USE_AMP=False

history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[],
           'margin_loss':[], 'recon_loss':[]}


def run_epoch(loader, train=True):
    model.train(train)
    total_loss = total_correct = total_samples = 0
    total_ml = total_rl = 0.0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

            # Normalise to [0,1] for the decoder target
            imgs_target = (imgs - imgs.min()) / (imgs.max() - imgs.min() + 1e-8)

            # ── Mixed precision forward ────────────────────────────────────────
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                lengths, caps, recon = model(imgs, labels if train else None)
                loss, ml, rl = criterion(lengths, recon, labels, imgs_target)

            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()

            preds          = lengths.argmax(dim=1)
            total_correct  += (preds == labels).sum().item()
            total_samples  += labels.size(0)
            total_loss     += loss.item() * labels.size(0)
            total_ml       += ml          * labels.size(0)
            total_rl       += rl          * labels.size(0)

    n = total_samples
    return total_loss/n, total_correct/n, total_ml/n, total_rl/n


print("Training utilities ready ✓")

In [ ]:
best_val_acc = 0.0

print(f"{'Epoch':>6} | {'Tr Loss':>8} | {'Tr Acc':>7} | {'Vl Loss':>8} | {'Vl Acc':>7} | {'LR':>9}")
print("-" * 62)

for epoch in range(1, cfg.EPOCHS + 1):
    tr_loss, tr_acc, tr_ml, tr_rl = run_epoch(train_loader, train=True)
    vl_loss, vl_acc, vl_ml, vl_rl = run_epoch(val_loader,   train=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)
    history['margin_loss'].append(tr_ml)
    history['recon_loss'].append(tr_rl)

    tag = ""
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), cfg.SAVE_PATH)
        tag = " ← best"

    lr_now = optimizer.param_groups[0]['lr']
    print(f"{epoch:>6} | {tr_loss:>8.4f} | {tr_acc:>6.2%} | "
          f"{vl_loss:>8.4f} | {vl_acc:>6.2%} | {lr_now:>9.2e}{tag}")

print(f"\nDone. Best val accuracy: {best_val_acc:.2%}")

---
## Step 11 — Evaluation & Metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
e = range(1, len(history['train_loss']) + 1)

axes[0].plot(e, history['train_loss'], label='Train'); axes[0].plot(e, history['val_loss'], label='Val')
axes[0].set_title('Total Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(e, history['train_acc'], label='Train'); axes[1].plot(e, history['val_acc'], label='Val')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].set_xlabel('Epoch')

plt.suptitle('FracCaps Lite — Training History', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
model.load_state_dict(torch.load(cfg.SAVE_PATH, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc='Testing'):
        lengths, _, _ = model(imgs.to(DEVICE))
        all_preds.append(lengths.argmax(dim=1).cpu())
        all_labels.append(labels)

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()
test_acc   = (all_preds == all_labels).mean()
print(f"Test Accuracy: {test_acc:.4f} ({test_acc:.2%})")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

In [ ]:
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Normalized Confusion Matrix — FracCaps Lite')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

---
## Step 12 — Visualizations
### 12.1 — Reconstructions

In [ ]:
model.eval()
imgs_sample, labels_sample = next(iter(test_loader))
imgs_sample   = imgs_sample[:8].to(DEVICE)
labels_sample = labels_sample[:8].to(DEVICE)

with torch.no_grad():
    lengths, caps, recon = model(imgs_sample, labels_sample)

def denormalize(t):
    if cfg.DATASET == 'MNIST':
        return (t * 0.3081 + 0.1307).clamp(0,1)
    mean = torch.tensor([0.4914,0.4822,0.4465]).view(3,1,1)
    std  = torch.tensor([0.2023,0.1994,0.2010]).view(3,1,1)
    return (t * std + mean).clamp(0,1)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    orig = denormalize(imgs_sample[i].cpu())
    rec  = recon[i].cpu().clamp(0,1)
    # squeeze for greyscale
    orig_np = orig.squeeze().numpy()
    rec_np  = rec.squeeze().numpy()
    cmap = 'gray' if cfg.IMG_CHANNELS == 1 else None

    axes[0,i].imshow(orig_np, cmap=cmap); axes[0,i].set_title(CLASS_NAMES[labels_sample[i]], fontsize=8); axes[0,i].axis('off')
    pred = lengths[i].argmax().item()
    col  = 'green' if pred == labels_sample[i].item() else 'red'
    axes[1,i].imshow(rec_np, cmap=cmap); axes[1,i].set_title(CLASS_NAMES[pred], fontsize=8, color=col); axes[1,i].axis('off')

plt.suptitle('Original (top) vs Reconstruction (bottom) — green=correct', y=1.01)
plt.tight_layout(); plt.show()

### 12.2 — Capsule Activation Heatmap

In [ ]:
with torch.no_grad():
    lengths_viz, _, _ = model(imgs_sample)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(lengths_viz.cpu().numpy(), annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=CLASS_NAMES,
            yticklabels=[f'Img{i} ({CLASS_NAMES[labels_sample[i]]})' for i in range(8)], ax=ax)
ax.set_title('Class capsule lengths (higher = more confident)')
plt.tight_layout(); plt.show()

### 12.3 — Capsule Dimension Perturbation
Perturb each capsule dimension to reveal what pose properties it encodes.

In [ ]:
img_single   = imgs_sample[[0]]
label_single = labels_sample[[0]]

with torch.no_grad():
    _, caps_single, _ = model(img_single, label_single)

pred_class = lengths_viz[0].argmax().item()
caps_vec   = caps_single[0, pred_class].clone()

n_dims  = min(6, cfg.CLASS_CAPS_DIM)
n_steps = 7
offsets = torch.linspace(-0.5, 0.5, n_steps)
cmap    = 'gray' if cfg.IMG_CHANNELS == 1 else None

fig, axes = plt.subplots(n_dims, n_steps, figsize=(n_steps * 1.6, n_dims * 1.6))
for d in range(n_dims):
    for s, delta in enumerate(offsets):
        perturbed         = caps_vec.clone(); perturbed[d] += delta
        caps_mod          = caps_single.clone(); caps_mod[0, pred_class] = perturbed
        with torch.no_grad():
            recon_p = model.decoder(caps_mod, label_single)
        axes[d,s].imshow(recon_p[0].cpu().clamp(0,1).squeeze().numpy(), cmap=cmap)
        axes[d,s].axis('off')
        if s == 0: axes[d,s].set_ylabel(f'dim {d}', fontsize=8)

plt.suptitle(f'Capsule perturbation — class: {CLASS_NAMES[pred_class]} | −0.5 → +0.5', fontsize=10)
plt.tight_layout(); plt.show()

---
## When You're Ready to Scale Back Up

Once thermals are under control, you can gradually re-enable heavier settings:

| Step | Change | Expected temp delta |
|------|--------|---------------------|
| 1 | `DATASET = 'CIFAR10'`, set `IMG_SIZE=32`, `IMG_CHANNELS=3` | +5–10°C |
| 2 | `BASE_CHANNELS = 32` | +5°C |
| 3 | `FRACTAL_DEPTH = 3` | +8–12°C |
| 4 | `PRIMARY_CAPS_NUM = 32`, `CLASS_CAPS_DIM = 16` | +3–5°C |
| 5 | `ROUTING_ITERATIONS = 3` | +2–3°C |
| 6 | `BATCH_SIZE = 64`, `EPOCHS = 50` | longer run, same peak temp |

Also consider: **add a fan curve profile** in your system BIOS or use a tool like `fancontrol` (Linux) or Macs Fan Control (macOS) to keep the CPU under 85°C during training.